# Notebook 3: Data Cleaning and Transformation

This notebook focuses on cleaning and transforming the Developer Survey dataset to prepare it for analysis and visualization in PowerBI. Building on the exploratory data analysis (EDA) from Notebook 2, we apply necessary and sensible transformations to a subset of relevant columns, addressing issues such as null values, inconsistent formats, and duplicates. The cleaned dataset will be loaded into a new PostgreSQL table called `clean_survey`, optimized for downstream use.

### Objectives
- Clean and standardize data in the specified relevant columns.
- Handle missing values and duplicates appropriately.
- Ensure data types and formats are suitable for PowerBI.
- Create and populate the `clean_survey` table in PostgreSQL.

## Setup

We import the required libraries for data manipulation and database interaction, then establish a connection to the PostgreSQL database using environment variables for security.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, Boolean
from sqlalchemy_utils import database_exists, create_database
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get environment variables
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# Build database connection URL
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create the database engine
engine = create_engine(DATABASE_URL)
print('Database connection established successfully.')

ValueError: invalid literal for int() with base 10: 'None'

## Data Loading

We load the dataset from the `raw_survey` table into a Pandas DataFrame for processing.

In [2]:
# Load data from raw_survey table
query = "SELECT * FROM raw_survey"
try:
    df = pd.read_sql(query, engine)
    print(f"Data loaded successfully. Rows: {len(df)}, Columns: {len(df.columns)}")
except Exception as e:
    print(f"Error loading data: {e}")
    raise

Data loaded successfully. Rows: 98855, Columns: 129


## Column Selection

We filter the dataset to include only the relevant columns specified for analysis and visualization in PowerBI.

In [3]:
# Define relevant columns
relevant_columns = [
    'Respondent',  # Unique identifier
    # Demographic
    'Country', 'Gender', 'Age', 'FormalEducation', 'RaceEthnicity',
    # Professional Experience
    'DevType', 'CompanySize', 'Employment', 'YearsCoding', 'YearsCodingProf',
    # Technologies and Tools
    'LanguageWorkedWith', 'DatabaseWorkedWith',
    'IDE', 'OperatingSystem',
    # Salary and Compensation
    'ConvertedSalary',
    # Job Satisfaction
    'JobSatisfaction', 'CareerSatisfaction', 'HopeFiveYears'
]

# Create a new DataFrame with selected columns
df_clean = df[relevant_columns].copy()
print(f"Filtered to relevant columns. New shape: {df_clean.shape}")

# Verify unique values and frequencies in Gender (before transformation)
print('Unique values in Gender:', df_clean['Gender'].unique())
print('\nFrequencies of Gender values:')
print(df_clean['Gender'].value_counts())

Filtered to relevant columns. New shape: (98855, 19)
Unique values in Gender: ['Male' None 'Female' 'Non-binary, genderqueer, or gender non-conforming'
 'Transgender' 'Female;Transgender' 'Female;Male'
 'Male;Non-binary, genderqueer, or gender non-conforming'
 'Transgender;Non-binary, genderqueer, or gender non-conforming'
 'Female;Non-binary, genderqueer, or gender non-conforming'
 'Female;Male;Transgender;Non-binary, genderqueer, or gender non-conforming'
 'Female;Transgender;Non-binary, genderqueer, or gender non-conforming'
 'Male;Transgender' 'Female;Male;Transgender'
 'Female;Male;Non-binary, genderqueer, or gender non-conforming'
 'Male;Transgender;Non-binary, genderqueer, or gender non-conforming']

Frequencies of Gender values:
Gender
Male                                                                         59458
Female                                                                        4025
Non-binary, genderqueer, or gender non-conforming                              2

## Data Transformations

We apply transformations to each category of columns to ensure consistency and usability. Each transformation is documented with its purpose.

### Demographic Columns

- **Country**: Standardize text and handle nulls.
- **Gender**: Simplify to main categories and handle nulls.
- **Age**: Convert ranges to numeric midpoints.
- **FormalEducation**: Standardize categories.
- **RaceEthnicity**: Simplify and standardize.

In [4]:
# Transform demographic columns
# Country: Convert to title case and replace nulls with 'Unknown'
df_clean['Country'] = df_clean['Country'].str.title().fillna('Unknown')

# Gender: Simplify to Male, Female, Other, or Prefer not to say
def simplify_gender(gender):
    if pd.isna(gender) or gender == '' or gender is None:
        return 'Prefer not to say'
    gender_list = [g.strip().lower() for g in str(gender).split(';')]
    if any('female' in g for g in gender_list):
        return 'Female'
    elif any('male' in g for g in gender_list):
        return 'Male'
    elif any('transgender' in g or 'non-binary' in g or 'genderqueer' in g for g in gender_list):
        return 'Other'
    return 'Other'
df_clean['Gender'] = df_clean['Gender'].apply(simplify_gender)

# Age: Convert ranges to midpoints and fill nulls with median
age_mapping = {
    'Under 18 years old': 16,
    '18 - 24 years old': 21,
    '25 - 34 years old': 29.5,
    '35 - 44 years old': 39.5,
    '45 - 54 years old': 49.5,
    '55 - 64 years old': 59.5,
    '65 years or older': 70
}
# Map to numeric values first
df_clean['Age'] = df_clean['Age'].map(age_mapping)
# Fill NaN with median of mapped values
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# FormalEducation: Strip whitespace and replace nulls with 'Unknown'
df_clean['FormalEducation'] = df_clean['FormalEducation'].str.strip().fillna('Unknown')

# RaceEthnicity: Take first category and handle nulls
def simplify_race(race):
    if pd.isna(race):
        return 'Prefer not to say'
    return race.split(';')[0] if ';' in race else race
df_clean['RaceEthnicity'] = df_clean['RaceEthnicity'].apply(simplify_race)

print('Demographic columns transformed successfully.')

# Verify Gender transformation
print('Unique values in Gender:', df_clean['Gender'].unique())
print('\nFrequencies of Gender values:')
print(df_clean['Gender'].value_counts())

Demographic columns transformed successfully.
Unique values in Gender: ['Male' 'Prefer not to say' 'Female' 'Other']

Frequencies of Gender values:
Gender
Male                 59620
Prefer not to say    34386
Female                4409
Other                  440
Name: count, dtype: int64


### Professional Experience Columns

- **DevType**: Keep as semicolon-separated text for flexibility.
- **CompanySize**: Convert to categorical ranges.
- **Employment**: Standardize categories.
- **YearsCoding** and **YearsCodingProf**: Convert ranges to midpoints.

In [5]:
# Transform professional experience columns

# DevType: Replace nulls with 'Unknown'
df_clean['DevType'] = df_clean['DevType'].fillna('Unknown')

# CompanySize: Convert ranges to standardized format
company_size_mapping = {
    'Fewer than 10 employees': '1-9',
    '10 to 19 employees': '10-19',
    '20 to 99 employees': '20-99',
    '100 to 499 employees': '100-499',
    '500 to 999 employees': '500-999',
    '1,000 to 4,999 employees': '1000-4999',
    '5,000 to 9,999 employees': '5000-9999',
    '10,000 or more employees': '10000+'
}
df_clean['CompanySize'] = df_clean['CompanySize'].map(company_size_mapping).fillna('Unknown')

# Employment: Strip whitespace and replace nulls with 'Unknown'
df_clean['Employment'] = df_clean['Employment'].str.strip().fillna('Unknown')

# YearsCoding and YearsCodingProf: Convert ranges to midpoints
years_mapping = {
    '0-2 years': 1,
    '3-5 years': 4,
    '6-8 years': 7,
    '9-11 years': 10,
    '12-14 years': 13,
    '15-17 years': 16,
    '18-20 years': 19,
    '21-23 years': 22,
    '24-26 years': 25,
    '27-29 years': 28,
    '30 or more years': 35
}
# Transform YearsCoding and fill nulls with median
df_clean['YearsCoding'] = df_clean['YearsCoding'].map(years_mapping)
df_clean['YearsCoding'] = df_clean['YearsCoding'].fillna(df_clean['YearsCoding'].median())

# Transform YearsCodingProf and fill nulls with 0
df_clean['YearsCodingProf'] = df_clean['YearsCodingProf'].map(years_mapping).fillna(0)

print('Professional experience columns transformed successfully.')

Professional experience columns transformed successfully.


In [6]:
df_clean.head()

,Respondent,Country,Gender,Age,FormalEducation,RaceEthnicity,DevType,CompanySize,Employment,YearsCoding,YearsCodingProf,LanguageWorkedWith,DatabaseWorkedWith,IDE,OperatingSystem,ConvertedSalary,JobSatisfaction,CareerSatisfaction,HopeFiveYears
0,438,United Kingdom,Male,29.5,"Bachelor’s degree (BA, BS, B.Eng., etc.)",White or of European descent,Back-end developer;Database administrator;Fron...,10-19,Employed full-time,16.0,7.0,Java;JavaScript;Lua;SQL;HTML;CSS,SQL Server,Eclipse;Notepad++,Windows,65285.0,Extremely satisfied,Extremely satisfied,Doing the same work
1,439,Switzerland,Prefer not to say,29.5,"Professional degree (JD, MD, etc.)",Prefer not to say,Back-end developer;Database administrator;Desk...,1000-4999,Employed full-time,7.0,4.0,Assembly;C;C++;Haskell;Java;JavaScript;PHP;Pyt...,MongoDB;MySQL;Oracle;MariaDB;Microsoft Azure (...,Atom;NetBeans;Notepad++;Vim;Visual Studio Code,Windows,NaN,Slightly dissatisfied,Moderately satisfied,Working in a different or more specialized tec...
2,440,China,Male,29.5,Some college/university study without earning ...,East Asian,Mobile developer,20-99,Employed full-time,4.0,4.0,C;JavaScript;Objective-C;SQL;Swift;HTML;CSS;Ba...,None,Sublime Text;Vim;Xcode,MacOS,NaN,Moderately satisfied,Slightly satisfied,Working in a different or more specialized tec...
3,443,United States,Prefer not to say,29.5,"Bachelor’s degree (BA, BS, B.Eng., etc.)",Prefer not to say,Front-end developer,100-499,Employed full-time,13.0,10.0,C#;JavaScript;PHP;Ruby;SQL;TypeScript;HTML;CSS...,SQL Server;MySQL,Visual Studio Code,MacOS,75000.0,Moderately satisfied,Slightly satisfied,Working in a different or more specialized tec...
4,12764,Colombia,Prefer not to say,29.5,"Bachelor’s degree (BA, BS, B.Eng., etc.)",Prefer not to say,Back-end developer,10-19,Employed full-time,19.0,0.0,C++;Clojure;Java;HTML;CSS,MySQL;Oracle,Eclipse;Emacs,Windows,NaN,None,None,None


### Technologies and Tools Columns

- **LanguageWorkedWith**, **LanguageDesireNextYear**, etc.: Retain semicolon-separated format, handle nulls.
- **OperatingSystem**: Standardize categories.

In [7]:
# Analyze and transform technologies and tools columns

# OperatingSystem: Standardize and handle nulls
df_clean['OperatingSystem'] = df_clean['OperatingSystem'].str.strip().fillna('Unknown')

# Define tech columns with multiple values
tech_columns = ['LanguageWorkedWith', 'DatabaseWorkedWith', 'IDE']

# Analyze unique values in each tech column
for col in tech_columns:
    # Split values by semicolon, flatten the list, and remove duplicates
    unique_values = set()
    df_clean[col].dropna().str.split(';').apply(lambda x: unique_values.update(x))
    print(f"Unique values in {col}: {sorted(unique_values)}")
    print(f"Total unique options in {col}: {len(unique_values)}\n")

print('OperatingSystem transformed and tech columns analyzed successfully.')

Unique values in LanguageWorkedWith: ['Assembly', 'Bash/Shell', 'C', 'C#', 'C++', 'CSS', 'Clojure', 'Cobol', 'CoffeeScript', 'Delphi/Object Pascal', 'Erlang', 'F#', 'Go', 'Groovy', 'HTML', 'Hack', 'Haskell', 'Java', 'JavaScript', 'Julia', 'Kotlin', 'Lua', 'Matlab', 'Objective-C', 'Ocaml', 'PHP', 'Perl', 'Python', 'R', 'Ruby', 'Rust', 'SQL', 'Scala', 'Swift', 'TypeScript', 'VB.NET', 'VBA', 'Visual Basic 6']
Total unique options in LanguageWorkedWith: 38

Unique values in DatabaseWorkedWith: ['Amazon DynamoDB', 'Amazon RDS/Aurora', 'Amazon Redshift', 'Apache HBase', 'Apache Hive', 'Cassandra', 'Elasticsearch', 'Google BigQuery', 'Google Cloud Storage', 'IBM Db2', 'MariaDB', 'Memcached', 'Microsoft Azure (Tables, CosmosDB, SQL, etc)', 'MongoDB', 'MySQL', 'Neo4j', 'Oracle', 'PostgreSQL', 'Redis', 'SQL Server', 'SQLite']
Total unique options in DatabaseWorkedWith: 21

Unique values in IDE: ['Android Studio', 'Atom', 'Coda', 'Eclipse', 'Emacs', 'IPython / Jupyter', 'IntelliJ', 'Komodo', 'Lig

In [8]:
# Analyze top 20 most used values in tech columns

# Define tech columns with multiple values
tech_columns = ['LanguageWorkedWith', 'DatabaseWorkedWith', 'IDE']

# Calculate and display top20 for each column
for col in tech_columns:
    # Split values by semicolon and explode into individual rows
    value_counts = (df_clean[col].dropna()
                    .str.split(';')
                    .explode()
                    .value_counts()
                    .head(20))  # Top 20 most frequent
    print(f"Top 20 most used in {col}:")
    print(value_counts.to_string())  # Formatted as a table-like output
    print()  # Extra line break between columns

print('Top 20 distributions calculated successfully.')

Top 20 most used in LanguageWorkedWith:
LanguageWorkedWith
JavaScript     54686
HTML           53628
CSS            50979
SQL            44670
Java           35521
Bash/Shell     31172
Python         30359
C#             26954
PHP            24071
C++            19872
C              18042
TypeScript     13626
Ruby            7911
Swift           6310
Assembly        5760
Go              5532
Objective-C     5510
VB.NET          5254
R               4813
Matlab          4564

Top 20 most used in DatabaseWorkedWith:
DatabaseWorkedWith
MySQL                                           38909
SQL Server                                      27293
PostgreSQL                                      21776
MongoDB                                         17183
SQLite                                          13036
Redis                                           11944
Elasticsearch                                    9312
MariaDB                                          8853
Oracle                       

In [9]:
# Apply encoding to tech columns with database type counters

# Define selected values for languages and IDEs
languages_to_keep = ['JavaScript', 'HTML', 'CSS', 'Python', 'Java', 'Bash/Shell',
                    'C#', 'C++', 'PHP', 'C', 'TypeScript', 'Ruby', 'Swift', 'Kotlin']
ides_to_keep = ['Visual Studio Code', 'Visual Studio', 'Notepad++', 'Sublime Text', 'Vim']

# Define database type mapping
db_type_mapping = {
    # SQL
    'MySQL': 'SQL', 'MariaDB': 'SQL', 'PostgreSQL': 'SQL', 'SQL Server': 'SQL',
    'SQLite': 'SQL', 'Oracle': 'SQL', 'IBM Db2': 'SQL', 'Amazon RDS/Aurora': 'SQL',
    # NoSQL
    'MongoDB': 'NoSQL', 'Cassandra': 'NoSQL', 'Neo4j': 'NoSQL', 'Amazon DynamoDB': 'NoSQL',
    'Apache HBase': 'NoSQL', 'Redis': 'NoSQL', 'Elasticsearch': 'NoSQL', 'Memcached': 'NoSQL',
    # Data Warehouse
    'Amazon Redshift': 'DataWarehouse', 'Google BigQuery': 'DataWarehouse', 'Apache Hive': 'DataWarehouse',
    # Other
    'Google Cloud Storage': 'Other', 'Microsoft Azure (Tables, CosmosDB, SQL, etc)': 'Other'
}

# Function for one-hot encoding (Languages and IDEs)
def one_hot_encode_column(df, column, values_to_keep, prefix):
    exploded = df[column].dropna().str.split(';').explode()
    dummies = pd.get_dummies(exploded).reindex(columns=values_to_keep, fill_value=0)
    encoded = dummies.groupby(level=0).sum().reindex(df.index, fill_value=0)
    encoded.columns = [f"{prefix}_{col.replace(' ', '_').replace('/', '_')}" for col in encoded.columns]
    other_count = exploded[~exploded.isin(values_to_keep)].groupby(level=0).size().reindex(df.index, fill_value=0)
    encoded[f"{prefix}_Other"] = other_count
    return pd.concat([df, encoded], axis=1)

# Function for database type counters
def encode_database_types(df, column, type_mapping):
    exploded = df[column].dropna().str.split(';').explode()
    # Map each database to its type
    db_types = exploded.map(type_mapping)
    # Count occurrences of each type per row
    type_counts = pd.get_dummies(db_types).groupby(level=0).sum().reindex(df.index, fill_value=0)
    # Rename columns with prefix
    type_counts.columns = [f"DB_{col}" for col in type_counts.columns]
    return pd.concat([df, type_counts], axis=1)

# Apply encoding
df_clean = one_hot_encode_column(df_clean, 'LanguageWorkedWith', languages_to_keep, 'Lang')
df_clean = encode_database_types(df_clean, 'DatabaseWorkedWith', db_type_mapping)
df_clean = one_hot_encode_column(df_clean, 'IDE', ides_to_keep, 'IDE')

print('Tech columns transformed with database type counters successfully.')

Tech columns transformed with database type counters successfully.


In [20]:
#df_clean.filter(like='Lang_').head()
df_clean.filter(like='DB_').head()
#df_clean.filter(like='IDE').head()

,DB_DataWarehouse,DB_NoSQL,DB_Other,DB_SQL
0,0,0,0,1
1,0,1,1,3
2,0,0,0,0
3,0,0,0,2
4,0,0,0,2


### Salary and Compensation Columns

- **Salary**: Clean non-numeric values, keep as string.
- **ConvertedSalary**: Cap outliers and handle nulls.
- **SalaryType** and **CurrencySymbol**: Standardize and handle nulls.

In [11]:
# Transform ConvertedSalary and clean up tech columns

# Cap ConvertedSalary at 99th percentile and fill nulls
salary_99th = df_clean['ConvertedSalary'].quantile(0.99)
df_clean['ConvertedSalary'] = df_clean['ConvertedSalary'].clip(upper=salary_99th)
df_clean['ConvertedSalary'] = df_clean['ConvertedSalary'].fillna(df_clean['ConvertedSalary'].median())

# Drop original tech columns with nulls
tech_columns_to_drop = ['LanguageWorkedWith', 'DatabaseWorkedWith', 'IDE']
df_clean = df_clean.drop(columns=tech_columns_to_drop)

print('ConvertedSalary transformed and original tech columns dropped successfully.')

ConvertedSalary transformed and original tech columns dropped successfully.


### Job Satisfaction Columns

- **JobSatisfaction**, **CareerSatisfaction**, **HopeFiveYears**: Standardize and handle nulls.

In [12]:
# Transform job satisfaction columns

# Define columns
satisfaction_columns = ['JobSatisfaction', 'CareerSatisfaction', 'HopeFiveYears']

# Standardize and handle nulls
for col in satisfaction_columns:
    df_clean[col] = df_clean[col].str.strip().str.title().fillna('Unknown')

# Print unique values for verification
for col in satisfaction_columns:
    print(f"Unique values in {col}: {df_clean[col].unique()}")

print('Job satisfaction columns transformed successfully.')

Unique values in JobSatisfaction: ['Extremely Satisfied' 'Slightly Dissatisfied' 'Moderately Satisfied'
 'Unknown' 'Moderately Dissatisfied' 'Extremely Dissatisfied'
 'Slightly Satisfied' 'Neither Satisfied Nor Dissatisfied']
Unique values in CareerSatisfaction: ['Extremely Satisfied' 'Moderately Satisfied' 'Slightly Satisfied'
 'Unknown' 'Moderately Dissatisfied' 'Slightly Dissatisfied'
 'Extremely Dissatisfied' 'Neither Satisfied Nor Dissatisfied']
Unique values in HopeFiveYears: ['Doing The Same Work'
 "Working In A Different Or More Specialized Technical Role Than The One I'M In Now"
 'Unknown' 'Retirement'
 'Working As A Founder Or Co-Founder Of My Own Company'
 'Working As An Engineering Manager Or Other Functional Manager'
 'Working As A Product Manager Or Project Manager'
 'Working In A Career Completely Unrelated To Software Development']
Job satisfaction columns transformed successfully.


## Null Handling

Most nulls have been addressed in the transformations above. We verify the remaining null counts to ensure completeness.

In [13]:
# Check for remaining nulls
null_counts = df_clean.isnull().sum()
print('Remaining null counts:\n', null_counts[null_counts > 0])

Remaining null counts:
 Series([], dtype: int64)


In [14]:
# Check for duplicates
duplicates = df_clean.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

Number of duplicate rows: 0


## Table Creation

We define and create the `clean_survey` table in PostgreSQL with appropriate data types for each column.

In [15]:
# Define and create clean_survey table

# Define metadata
metadata = MetaData()

# Define clean_survey table
clean_survey = Table(
    'clean_survey', metadata,
    Column('Respondent', Integer, primary_key=True),  # Unique identifier
    # Demographic
    Column('Country', String),
    Column('Gender', String),
    Column('Age', Float),
    Column('FormalEducation', String),
    Column('RaceEthnicity', String),
    # Professional Experience
    Column('DevType', String),
    Column('CompanySize', String),
    Column('Employment', String),
    Column('YearsCoding', Float),
    Column('YearsCodingProf', Float),
    # Technologies and Tools
    # Languages
    Column('Lang_JavaScript', Integer),
    Column('Lang_HTML', Integer),
    Column('Lang_CSS', Integer),
    Column('Lang_Python', Integer),
    Column('Lang_Java', Integer),
    Column('Lang_Bash_Shell', Integer),
    Column('Lang_CSharp', Integer),
    Column('Lang_CPlusPlus', Integer),
    Column('Lang_PHP', Integer),
    Column('Lang_C', Integer),
    Column('Lang_TypeScript', Integer),
    Column('Lang_Ruby', Integer),
    Column('Lang_Swift', Integer),
    Column('Lang_Kotlin', Integer),
    Column('Lang_Other', Integer),
    # Databases
    Column('DB_SQL', Integer),
    Column('DB_NoSQL', Integer),
    Column('DB_DataWarehouse', Integer),
    Column('DB_Other', Integer),
    # IDEs
    Column('IDE_Visual_Studio_Code', Integer),
    Column('IDE_Visual_Studio', Integer),
    Column('IDE_NotepadPlusPlus', Integer),
    Column('IDE_Sublime_Text', Integer),
    Column('IDE_Vim', Integer),
    Column('IDE_Other', Integer),
    Column('OperatingSystem', String),
    # Salary and Compensation
    Column('ConvertedSalary', Float),
    # Job Satisfaction
    Column('JobSatisfaction', String),
    Column('CareerSatisfaction', String),
    Column('HopeFiveYears', String)
)

# Create table in database
metadata.create_all(engine)
print('Table clean_survey created successfully.')

Table clean_survey created successfully.


## Data Loading

We load the cleaned DataFrame into the `clean_survey` table, replacing any existing data.

In [16]:
# Load the cleaned data into clean_survey
# df_clean.to_sql('clean_survey', engine, if_exists='replace', index=False, method='multi')
# print('Cleaned data loaded into clean_survey table successfully.')

In [18]:
# Load data into clean_survey table

from sqlalchemy import text  # Import text for raw SQL

try:
    # Load DataFrame into clean_survey
    df_clean.to_sql(
        'clean_survey',
        engine,
        if_exists='replace',  # Replace table if it exists
        index=False,          # Do not write DataFrame index
        method='multi',       # Use multi-row inserts for performance
        chunksize=1000        # Process in batches of 1000 rows
    )
    
    # Verify row count in database
    with engine.connect() as connection:
        row_count = connection.execute(text("SELECT COUNT(*) FROM clean_survey")).scalar()
        print(f"Cleaned data loaded successfully. {row_count} rows inserted into clean_survey.")

except Exception as e:
    print(f"Error loading data into clean_survey: {e}")
    raise

Cleaned data loaded successfully. 98855 rows inserted into clean_survey.


## Verification

We query the first 5 rows of the `clean_survey` table to confirm the data was loaded correctly.

In [22]:
print(f"Rows in df_clean: {len(df_clean)}")
# Luego del to_sql
print(f"Rows in database: {row_count}")
assert len(df_clean) == row_count, "Row counts do not match!"

Rows in df_clean: 98855
Rows in database: 98855
